# Classification rekomendasi usaha sintetis

Mengevaluasi classifier dengan holdout berbasis stasiun. Label sintetis tidak membuktikan kategori usaha terbaik di dunia nyata.

Semua input dan output saat ini adalah prototipe sintetis. Hasil tidak boleh dianggap sebagai observasi lapangan atau rekomendasi bisnis/investasi produksi.

In [1]:
from pathlib import Path

def find_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'ml' / 'DATASET_CATALOG.md').exists():
            return candidate
    raise FileNotFoundError('Jalankan notebook dari dalam repository TCI')

ROOT = find_root()
ML_ROOT = ROOT / 'ml'
SEED = 20260911
STATUS = 'synthetic_prototype'
print(f'Project root: {ROOT}')

Project root: C:\Users\axels\Axel Documents\Documents\BINUS\Lomba\MAPID WebGIS (Top 50)\App\TCI


In [ ]:

import json
import pickle
import platform
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, f1_score, top_k_accuracy_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

feature_dir = ML_ROOT / "business_classification"
data_path = feature_dir / "data" / "location_opportunity_training.csv"
models_dir, outputs_dir = feature_dir / "models", feature_dir / "outputs"
models_dir.mkdir(parents=True, exist_ok=True)
outputs_dir.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(data_path)
df["month"] = pd.to_datetime(df.period + "-01").dt.month

features = ["buffer_radius_m", "estimated_daily_footfall", "passenger_volume_monthly", "food_poi_count", "beverage_poi_count", "hobby_poi_count", "bookstore_poi_count", "other_retail_poi_count", "office_activity_index", "residential_activity_index", "education_activity_index", "pedestrian_access_index", "estimated_rental_index_rp_thousand_m2_month", "month"]
target = "recommended_business_category"
duplicates = int(df.record_id.duplicated().sum())
missing = {column: int(value) for column, value in df[features + [target]].isna().sum().items() if value}
invalid_status = int((df.source_status != STATUS).sum())
quality_passed = not (duplicates or missing or invalid_status)
quality_report = {"passed": quality_passed, "record_count": len(df), "station_count": int(df.station_code.nunique()), "duplicate_record_ids": duplicates, "missing_values": missing, "class_distribution": {str(key): int(value) for key, value in df[target].value_counts().items()}, "invalid_source_status": invalid_status}
(outputs_dir / "data_quality_report.json").write_text(json.dumps(quality_report, indent=2), encoding="utf-8")
if not quality_passed:
    raise ValueError("Classification quality gate failed; outputs are unavailable")

all_classes = sorted(df[target].unique())
splitter = GroupShuffleSplit(n_splits=50, test_size=0.2, random_state=SEED)
train_idx = test_idx = None
for candidate_train, candidate_test in splitter.split(df[features], df[target], groups=df.station_code):
    if set(df.iloc[candidate_train][target]) == set(all_classes) and set(df.iloc[candidate_test][target]) == set(all_classes):
        train_idx, test_idx = candidate_train, candidate_test
        break
if train_idx is None:
    raise ValueError("Unable to make a station-group holdout containing every category")
train, test = df.iloc[train_idx], df.iloc[test_idx]
X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

logistic = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=4000, class_weight="balanced", random_state=SEED))])
forest = Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", RandomForestClassifier(n_estimators=500, min_samples_leaf=2, class_weight="balanced_subsample", random_state=SEED, n_jobs=-1))])
candidates = {"logistic_regression": logistic, "random_forest": forest}
candidate_metrics = {}
for name, pipeline in candidates.items():
    pipeline.fit(X_train, y_train)
    prediction = pipeline.predict(X_test)
    probabilities = pipeline.predict_proba(X_test)
    labels = pipeline.named_steps["model"].classes_
    candidate_metrics[name] = {
        "macro_f1": f1_score(y_test, prediction, average="macro"),
        "balanced_accuracy": balanced_accuracy_score(y_test, prediction),
        "top_3_accuracy": top_k_accuracy_score(y_test, probabilities, k=min(3, len(labels)), labels=labels),
        "confusion_matrix": confusion_matrix(y_test, prediction, labels=all_classes).tolist(),
        "confusion_matrix_labels": all_classes,
    }
selected_name = max(candidates, key=lambda name: (candidate_metrics[name]["macro_f1"], candidate_metrics[name]["balanced_accuracy"]))
selected_pipeline = candidates[selected_name]
selected_pipeline.fit(df[features], df[target])

latest = df.sort_values("period").groupby(["station_code", "location_id", "buffer_radius_m"], as_index=False).tail(1).copy()
probabilities = selected_pipeline.predict_proba(latest[features])
classes = selected_pipeline.named_steps["model"].classes_
order = np.argsort(probabilities, axis=1)[:, ::-1]
latest["recommended_category"] = [classes[indexes[0]] for indexes in order]
latest["recommendation_confidence"] = [round(float(row[indexes[0]]), 4) for row, indexes in zip(probabilities, order)]
latest["top_3_categories"] = [json.dumps([classes[index] for index in indexes[:3]], ensure_ascii=False) for indexes in order]

if selected_name == "random_forest":
    importance_values = selected_pipeline.named_steps["model"].feature_importances_
else:
    importance_values = np.abs(selected_pipeline.named_steps["model"].coef_).mean(axis=0)
importance = dict(sorted(zip(features, map(float, importance_values)), key=lambda item: item[1], reverse=True))
reason_features = list(importance)[:3]
latest["feature_reason"] = "Fitur model paling berpengaruh: " + ", ".join(reason_features) + "."
latest["source_status"] = STATUS
latest["recommendation_status"] = "prototype_only_not_validated_business_outcome"
output_columns = ["location_id", "latitude", "longitude", "station_code", "station_id", "station_name", "period", "buffer_radius_m", "recommended_category", "recommendation_confidence", "top_3_categories", "feature_reason", "recommendation_status", "source_status"]
latest[output_columns].to_csv(outputs_dir / "location_recommendations.csv", index=False)

artifact = {"pipeline": selected_pipeline, "features": features, "selected_model": selected_name, "classes": classes.tolist(), "source_status": STATUS, "warning": "Synthetic labels; not a validated best-business recommendation."}
with (models_dir / "business_recommender.pkl").open("wb") as file:
    pickle.dump(artifact, file)
metrics = {"selected_model": selected_name, "selection_metric": "station-group holdout macro F1", "split": {"strategy": "GroupShuffleSplit by station_code", "train_station_count": int(train.station_code.nunique()), "test_station_count": int(test.station_code.nunique()), "station_overlap": len(set(train.station_code) & set(test.station_code))}, "candidates": candidate_metrics, "global_feature_importance": importance, "source_status": STATUS, "limitation": "Labels are synthetic and imbalanced; metrics do not validate real business outcomes."}
(outputs_dir / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
(outputs_dir / "feature_schema.json").write_text(json.dumps({"features": features, "target": target, "identifier_fields": ["record_id", "location_id", "station_code", "station_id"], "geometry_fields": ["latitude", "longitude"], "grain": "location-station-month-radius", "published_radii_m": [500, 750, 1000], "excluded_from_training": ["synthetic_recommendation_confidence", "label_provenance", "source_status"]}, indent=2), encoding="utf-8")
manifest = {"run_at_utc": datetime.now(timezone.utc).isoformat(), "source": str(data_path.relative_to(ROOT)), "records": len(df), "model": selected_name, "seed": SEED, "python": platform.python_version(), "sklearn": sklearn.__version__, "source_status": STATUS, "limitations": "Synthetic proxies and labels; Hotspot Finder must not claim this is a validated best category."}
(outputs_dir / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps({"quality_passed": quality_passed, "selected_model": selected_name, "test_macro_f1": candidate_metrics[selected_name]["macro_f1"], "model": str(models_dir / 'business_recommender.pkl')}, indent=2))


{
  "quality_passed": true,
  "selected_model": "logistic_regression",
  "test_macro_f1": 0.7126578345471833,
  "model": "C:\\Users\\axels\\Axel Documents\\Documents\\BINUS\\Lomba\\MAPID WebGIS (Top 50)\\App\\TCI\\ml\\business_classification\\models\\business_recommender.pkl"
}
